# Notebook 5: Training Loops From Scratch

Build complete PyTorch training loops step by step -- no utility imports, just the raw pattern.

---

## Learning Objectives

By the end of this notebook you will be able to:

- Explain the four-step anatomy of a training loop: **forward -> loss -> backward -> step**
- Use `model.train()` and `model.eval()` correctly
- Understand why `optimizer.zero_grad()` is required
- Write complete training loops for binary classification, multi-class classification, and regression
- Add a validation loop and track metrics
- Plot training curves with matplotlib

## Prerequisites

- Notebooks 01-04 of this series (tensors, autograd, `nn.Module`, loss functions, optimizers)
- Basic Python and NumPy knowledge
- `torch`, `matplotlib` installed

## Table of Contents

1. [Anatomy of a Training Loop](#1)
2. [`model.train()` vs `model.eval()`](#2)
3. [Why `optimizer.zero_grad()`?](#3)
4. [Binary Classification -- Full Training Loop](#4)
5. [Multi-Class Classification -- Full Training Loop](#5)
6. [Logging Metrics During Training](#6)
7. [Adding a Validation Loop](#7)
8. [Plotting Training Curves](#8)
9. [Common Mistakes & Debugging Tips](#9)
10. [Exercises](#10)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")

<a id='1'></a>
## 1. Anatomy of a Training Loop

Every PyTorch training loop follows the same four-step pattern:

```
for epoch in range(num_epochs):
    for X_batch, y_batch in dataloader:
        # 1. Forward pass  -- compute predictions
        y_pred = model(X_batch)

        # 2. Compute loss
        loss = loss_fn(y_pred, y_batch)

        # 3. Backward pass -- compute gradients
        optimizer.zero_grad()   # clear old gradients
        loss.backward()         # accumulate new gradients

        # 4. Update weights
        optimizer.step()
```

**Why this order?**

| Step | What happens | PyTorch call |
|---|---|---|
| Forward | Input flows through the network to produce an output | `model(x)` |
| Loss | Scalar value measuring how wrong the prediction is | `loss_fn(pred, target)` |
| Backward | Gradients $\frac{\partial \mathcal{L}}{\partial w}$ are computed via chain rule | `loss.backward()` |
| Step | Weights updated: $w \leftarrow w - \eta \nabla_w \mathcal{L}$ | `optimizer.step()` |

<a id='2'></a>
## 2. `model.train()` vs `model.eval()`

Some layers behave **differently** during training vs inference:

| Layer | `model.train()` | `model.eval()` |
|---|---|---|
| `nn.Dropout` | Randomly zeroes elements | Passes all elements (scaled) |
| `nn.BatchNorm` | Uses batch statistics, updates running stats | Uses stored running stats |

**Rule of thumb:**
- Call `model.train()` before every training epoch
- Call `model.eval()` before every validation / test evaluation

In [ ]:
# Demonstration: effect of train() vs eval() on Dropout
torch.manual_seed(42)

dropout = nn.Dropout(p=0.5)
x = torch.ones(1, 10)

dropout.train()
print("train mode:", dropout(x))   # ~half the values zeroed, rest scaled by 2

dropout.eval()
print("eval mode: ", dropout(x))   # all values passed through unchanged

<a id='3'></a>
## 3. Why `optimizer.zero_grad()`?

PyTorch **accumulates** gradients by default.  Each call to `loss.backward()` **adds** to the existing `.grad` attribute rather than replacing it.

This is useful for:
- Gradient accumulation across mini-batches (simulating a larger batch size)

But if you forget to zero them out, gradients from previous steps leak into the current step, leading to **incorrect updates**.

In [ ]:
# Demonstration: gradient accumulation bug
torch.manual_seed(42)

w = torch.tensor([1.0], requires_grad=True)

# Simulate two backward passes WITHOUT zeroing
loss1 = (w * 2).sum()
loss1.backward()
print(f"After 1st backward: w.grad = {w.grad}")  # 2.0

loss2 = (w * 3).sum()
loss2.backward()
print(f"After 2nd backward (no zero_grad): w.grad = {w.grad}")  # 2.0 + 3.0 = 5.0  <-- wrong!

# Correct: zero first
w.grad.zero_()
loss3 = (w * 3).sum()
loss3.backward()
print(f"After zero_grad + backward: w.grad = {w.grad}")  # 3.0  <-- correct

<a id='4'></a>
## 4. Binary Classification -- Full Training Loop

We will:
1. Generate synthetic 2D data (two classes)
2. Build a small network
3. Train from scratch -- no utility functions, just the raw loop

In [ ]:
# ---- Synthetic binary data ----
torch.manual_seed(42)
np.random.seed(42)

n_samples = 500
# Class 0: centred around (-1, -1)
X0 = torch.randn(n_samples // 2, 2) + torch.tensor([-1.0, -1.0])
# Class 1: centred around (+1, +1)
X1 = torch.randn(n_samples // 2, 2) + torch.tensor([1.0, 1.0])

X_bin = torch.cat([X0, X1], dim=0)            # (500, 2)
y_bin = torch.cat([torch.zeros(n_samples // 2),
                   torch.ones(n_samples // 2)], dim=0)  # (500,)

# Shuffle
perm = torch.randperm(n_samples)
X_bin, y_bin = X_bin[perm], y_bin[perm]

print(f"X shape: {X_bin.shape}, y shape: {y_bin.shape}")
print(f"Class balance: {y_bin.mean().item():.2f} (should be ~0.50)")

In [ ]:
# Visualise
plt.figure(figsize=(6, 5))
mask0 = y_bin == 0
mask1 = y_bin == 1
plt.scatter(X_bin[mask0, 0], X_bin[mask0, 1], alpha=0.5, label="Class 0")
plt.scatter(X_bin[mask1, 0], X_bin[mask1, 1], alpha=0.5, label="Class 1")
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")
plt.title("Synthetic Binary Classification Data")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ---- Model ----
class BinaryClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),  # single logit output
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)  # (batch,)

torch.manual_seed(42)
model_bin = BinaryClassifier(input_dim=2)
print(model_bin)

In [ ]:
# ---- Full training loop (binary classification) ----
torch.manual_seed(42)

model_bin = BinaryClassifier(input_dim=2)
loss_fn = nn.BCEWithLogitsLoss()          # combines sigmoid + BCE
optimizer = optim.SGD(model_bin.parameters(), lr=0.1)

num_epochs = 100

train_losses = []
train_accs = []

for epoch in range(num_epochs):
    # ---------- Training ----------
    model_bin.train()                      # set to training mode

    # 1. Forward pass
    logits = model_bin(X_bin)              # raw logits

    # 2. Compute loss
    loss = loss_fn(logits, y_bin)

    # 3. Backward pass
    optimizer.zero_grad()                  # clear previous gradients
    loss.backward()                        # compute gradients

    # 4. Update weights
    optimizer.step()

    # ---------- Metrics ----------
    with torch.no_grad():
        preds = (torch.sigmoid(logits) >= 0.5).float()
        acc = (preds == y_bin).float().mean()

    train_losses.append(loss.item())
    train_accs.append(acc.item())

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d}/{num_epochs} | "
              f"Loss: {loss.item():.4f} | Acc: {acc.item():.4f}")

print("\nTraining complete.")

<a id='5'></a>
## 5. Multi-Class Classification -- Full Training Loop

Key differences from binary:
- Output layer has **C** neurons (one per class)
- Loss: `nn.CrossEntropyLoss` (expects **raw logits**, applies softmax internally)
- Targets: integer class indices (not one-hot)

In [ ]:
# ---- Synthetic multi-class data (3 classes) ----
torch.manual_seed(42)
np.random.seed(42)

n_per_class = 200
num_classes = 3

centres = torch.tensor([[0.0, 2.0],
                        [-1.5, -1.0],
                        [1.5, -1.0]])

X_multi_list, y_multi_list = [], []
for c in range(num_classes):
    X_c = torch.randn(n_per_class, 2) * 0.6 + centres[c]
    y_c = torch.full((n_per_class,), c, dtype=torch.long)
    X_multi_list.append(X_c)
    y_multi_list.append(y_c)

X_multi = torch.cat(X_multi_list, dim=0)
y_multi = torch.cat(y_multi_list, dim=0)

# Shuffle
perm = torch.randperm(len(y_multi))
X_multi, y_multi = X_multi[perm], y_multi[perm]

print(f"X shape: {X_multi.shape}, y shape: {y_multi.shape}")
print(f"Classes: {torch.unique(y_multi).tolist()}")

In [ ]:
# Visualise
plt.figure(figsize=(6, 5))
for c in range(num_classes):
    mask = y_multi == c
    plt.scatter(X_multi[mask, 0], X_multi[mask, 1], alpha=0.5, label=f"Class {c}")
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")
plt.title("Synthetic Multi-Class Data")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ---- Model ----
class MultiClassClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes),  # C output logits
        )

    def forward(self, x):
        return self.net(x)  # (batch, num_classes)

torch.manual_seed(42)
model_mc = MultiClassClassifier(input_dim=2, hidden_dim=32, num_classes=3)
print(model_mc)

In [ ]:
# ---- Full training loop (multi-class) ----
torch.manual_seed(42)

model_mc = MultiClassClassifier(input_dim=2, hidden_dim=32, num_classes=3)
loss_fn_mc = nn.CrossEntropyLoss()          # softmax + NLL inside
optimizer_mc = optim.Adam(model_mc.parameters(), lr=0.01)

num_epochs = 150
mc_losses, mc_accs = [], []

for epoch in range(num_epochs):
    model_mc.train()

    # 1. Forward
    logits = model_mc(X_multi)              # (N, 3)

    # 2. Loss
    loss = loss_fn_mc(logits, y_multi)      # targets are class indices

    # 3. Backward
    optimizer_mc.zero_grad()
    loss.backward()

    # 4. Step
    optimizer_mc.step()

    # Metrics
    with torch.no_grad():
        preds = logits.argmax(dim=1)
        acc = (preds == y_multi).float().mean()

    mc_losses.append(loss.item())
    mc_accs.append(acc.item())

    if (epoch + 1) % 30 == 0:
        print(f"Epoch {epoch+1:3d}/{num_epochs} | "
              f"Loss: {loss.item():.4f} | Acc: {acc.item():.4f}")

print("\nTraining complete.")

<a id='6'></a>
## 6. Logging Metrics During Training

Best practices:
- Store metrics in **lists** (or dicts) each epoch
- Print every N epochs to avoid flooding the console
- Track both **loss** and **task-specific metrics** (accuracy, F1, etc.)

Here is a reusable pattern:

In [ ]:
def train_one_epoch(model, X, y, loss_fn, optimizer):
    """Run one training epoch and return (loss, accuracy)."""
    model.train()

    logits = model(X)
    loss = loss_fn(logits, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    with torch.no_grad():
        preds = logits.argmax(dim=1)
        acc = (preds == y).float().mean()

    return loss.item(), acc.item()


# Usage
torch.manual_seed(42)
model_log = MultiClassClassifier(2, 32, 3)
opt_log = optim.Adam(model_log.parameters(), lr=0.01)

history = {"loss": [], "acc": []}
for epoch in range(100):
    loss_val, acc_val = train_one_epoch(model_log, X_multi, y_multi,
                                        nn.CrossEntropyLoss(), opt_log)
    history["loss"].append(loss_val)
    history["acc"].append(acc_val)

print(f"Final loss: {history['loss'][-1]:.4f}, Final acc: {history['acc'][-1]:.4f}")

<a id='7'></a>
## 7. Adding a Validation Loop

A validation loop differs from training in three ways:

1. **`model.eval()`** -- switches dropout / batchnorm to inference mode
2. **`torch.no_grad()`** -- disables gradient computation (saves memory & time)
3. **No optimizer step** -- we only measure, we do not update

In [ ]:
# ---- Train / Val split ----
torch.manual_seed(42)

n_total = len(X_multi)
n_train = int(0.8 * n_total)

indices = torch.randperm(n_total)
train_idx, val_idx = indices[:n_train], indices[n_train:]

X_train, y_train = X_multi[train_idx], y_multi[train_idx]
X_val,   y_val   = X_multi[val_idx],   y_multi[val_idx]

print(f"Train: {len(X_train)}, Val: {len(X_val)}")

In [ ]:
# ---- Training + Validation loop ----
torch.manual_seed(42)

model_val = MultiClassClassifier(2, 32, 3)
loss_fn_val = nn.CrossEntropyLoss()
optimizer_val = optim.Adam(model_val.parameters(), lr=0.01)

num_epochs = 150
history = {"train_loss": [], "train_acc": [],
           "val_loss": [],   "val_acc": []}

for epoch in range(num_epochs):
    # -------- TRAIN --------
    model_val.train()
    logits_t = model_val(X_train)
    loss_t = loss_fn_val(logits_t, y_train)

    optimizer_val.zero_grad()
    loss_t.backward()
    optimizer_val.step()

    with torch.no_grad():
        preds_t = logits_t.argmax(dim=1)
        acc_t = (preds_t == y_train).float().mean()

    # -------- VALIDATE --------
    model_val.eval()                       # <-- switch mode
    with torch.no_grad():                  # <-- no gradients
        logits_v = model_val(X_val)
        loss_v = loss_fn_val(logits_v, y_val)
        preds_v = logits_v.argmax(dim=1)
        acc_v = (preds_v == y_val).float().mean()

    # Log
    history["train_loss"].append(loss_t.item())
    history["train_acc"].append(acc_t.item())
    history["val_loss"].append(loss_v.item())
    history["val_acc"].append(acc_v.item())

    if (epoch + 1) % 30 == 0:
        print(f"Epoch {epoch+1:3d}/{num_epochs} | "
              f"Train Loss: {loss_t.item():.4f}  Acc: {acc_t.item():.4f} | "
              f"Val Loss: {loss_v.item():.4f}  Acc: {acc_v.item():.4f}")

print("\nTraining complete.")

<a id='8'></a>
## 8. Plotting Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs_range = range(1, num_epochs + 1)

# ---- Loss ----
axes[0].plot(epochs_range, history["train_loss"], label="Train")
axes[0].plot(epochs_range, history["val_loss"],   label="Val")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss Curves")
axes[0].legend()

# ---- Accuracy ----
axes[1].plot(epochs_range, history["train_acc"], label="Train")
axes[1].plot(epochs_range, history["val_acc"],   label="Val")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy Curves")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Also plot binary classification curves from Section 4
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_losses)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("BCE Loss")
axes[0].set_title("Binary Classification -- Loss")

axes[1].plot(train_accs)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Binary Classification -- Accuracy")

plt.tight_layout()
plt.show()

<a id='9'></a>
## 9. Common Mistakes & Debugging Tips

| Mistake | Symptom | Fix |
|---|---|---|
| Forgetting `optimizer.zero_grad()` | Loss oscillates or diverges | Add `optimizer.zero_grad()` before `loss.backward()` |
| Forgetting `model.eval()` in validation | Val metrics are noisy / wrong | Call `model.eval()` before validation |
| Forgetting `torch.no_grad()` in validation | OOM or slow validation | Wrap validation in `with torch.no_grad():` |
| Using `softmax` + `nn.CrossEntropyLoss` | Double softmax, loss never drops | `CrossEntropyLoss` already applies softmax internally |
| Using `sigmoid` + `nn.BCEWithLogitsLoss` | Same issue -- double sigmoid | `BCEWithLogitsLoss` already applies sigmoid internally |
| Wrong target dtype for `CrossEntropyLoss` | Runtime error | Targets must be `torch.long` (integer class indices) |
| Wrong target shape for `BCEWithLogitsLoss` | Broadcasting bug | Targets must match logit shape -- use `.squeeze()` / `.unsqueeze()` |
| Not shuffling data | Model memorises order | Shuffle or use `DataLoader(shuffle=True)` |
| Learning rate too high | Loss explodes (NaN) | Reduce `lr` by 10x |
| Learning rate too low | Loss barely decreases | Increase `lr` or train longer |

<a id='10'></a>
## 10. Exercises

### Exercise 1: Write a Training Loop for Regression

Use the synthetic data below.  Build a small network, train it with `nn.MSELoss()`, and plot the loss curve.

**Requirements:**
- Model: 2 hidden layers (64 units each), ReLU activation
- Optimizer: Adam, lr=0.01
- Train for 200 epochs
- Split data 80/20 into train/val
- Log train and val loss, then plot both curves

In [ ]:
# ---- Data for exercise ----
torch.manual_seed(42)
X_reg = torch.linspace(-3, 3, 300).unsqueeze(1)       # (300, 1)
y_reg = torch.sin(X_reg) + 0.2 * torch.randn_like(X_reg)  # noisy sine
y_reg = y_reg.squeeze(1)                               # (300,)

plt.scatter(X_reg.numpy(), y_reg.numpy(), s=10, alpha=0.6)
plt.title("Regression data: noisy sine")
plt.xlabel("x")
plt.ylabel("y")
plt.show()

# YOUR CODE HERE
# 1. Define model (nn.Module with 2 hidden layers of 64 units, ReLU, output dim=1)
# 2. Split data 80/20
# 3. Write training + validation loop for 200 epochs
# 4. Plot train and val MSE loss curves

### Exercise 1 -- Solution

<details>
<summary>Click to expand</summary>

In [ ]:
# ---- Solution: Regression Training Loop ----
torch.manual_seed(42)

# Model
class RegressionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

model_reg = RegressionNet()
loss_fn_reg = nn.MSELoss()
optimizer_reg = optim.Adam(model_reg.parameters(), lr=0.01)

# Train/Val split
n_total_reg = len(X_reg)
n_train_reg = int(0.8 * n_total_reg)
idx = torch.randperm(n_total_reg)
X_tr, y_tr = X_reg[idx[:n_train_reg]], y_reg[idx[:n_train_reg]]
X_vl, y_vl = X_reg[idx[n_train_reg:]], y_reg[idx[n_train_reg:]]

# Training loop
reg_history = {"train_loss": [], "val_loss": []}

for epoch in range(200):
    # Train
    model_reg.train()
    pred_t = model_reg(X_tr)
    loss_t = loss_fn_reg(pred_t, y_tr)

    optimizer_reg.zero_grad()
    loss_t.backward()
    optimizer_reg.step()

    # Validate
    model_reg.eval()
    with torch.no_grad():
        pred_v = model_reg(X_vl)
        loss_v = loss_fn_reg(pred_v, y_vl)

    reg_history["train_loss"].append(loss_t.item())
    reg_history["val_loss"].append(loss_v.item())

    if (epoch + 1) % 40 == 0:
        print(f"Epoch {epoch+1:3d}/200 | "
              f"Train MSE: {loss_t.item():.4f} | Val MSE: {loss_v.item():.4f}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(reg_history["train_loss"], label="Train")
axes[0].plot(reg_history["val_loss"],   label="Val")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE Loss")
axes[0].set_title("Regression -- Loss Curves")
axes[0].legend()

# Prediction overlay
model_reg.eval()
with torch.no_grad():
    X_plot = torch.linspace(-3, 3, 500).unsqueeze(1)
    y_plot = model_reg(X_plot)

axes[1].scatter(X_reg.numpy(), y_reg.numpy(), s=10, alpha=0.4, label="Data")
axes[1].plot(X_plot.numpy(), y_plot.numpy(), color="red", linewidth=2, label="Model")
axes[1].set_xlabel("x")
axes[1].set_ylabel("y")
axes[1].set_title("Regression -- Fit")
axes[1].legend()

plt.tight_layout()
plt.show()

</details>

---

### Exercise 2: Early Stopping

Modify the multi-class training loop from Section 7 to stop training when validation loss has not improved for **10 consecutive epochs** (patience = 10).  Print which epoch training stopped at.

<details>
<summary>Click for hint</summary>

Track `best_val_loss` and a `patience_counter`.  Increment the counter when val loss does not improve; reset it when it does.  Break out of the loop when the counter reaches 10.
</details>

In [ ]:
# YOUR CODE HERE
# Hint: copy the train+val loop from Section 7 and add early stopping logic

### Exercise 2 -- Solution

<details>
<summary>Click to expand</summary>

In [ ]:
# ---- Solution: Early Stopping ----
torch.manual_seed(42)

model_es = MultiClassClassifier(2, 32, 3)
loss_fn_es = nn.CrossEntropyLoss()
optimizer_es = optim.Adam(model_es.parameters(), lr=0.01)

max_epochs = 500
patience = 10
best_val_loss = float("inf")
patience_counter = 0

for epoch in range(max_epochs):
    # Train
    model_es.train()
    logits_t = model_es(X_train)
    loss_t = loss_fn_es(logits_t, y_train)
    optimizer_es.zero_grad()
    loss_t.backward()
    optimizer_es.step()

    # Validate
    model_es.eval()
    with torch.no_grad():
        logits_v = model_es(X_val)
        loss_v = loss_fn_es(logits_v, y_val)

    # Early stopping check
    if loss_v.item() < best_val_loss:
        best_val_loss = loss_v.item()
        patience_counter = 0
    else:
        patience_counter += 1

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1} | Train: {loss_t.item():.4f} | "
              f"Val: {loss_v.item():.4f} | Patience: {patience_counter}/{patience}")

    if patience_counter >= patience:
        print(f"\nEarly stopping at epoch {epoch+1}! "
              f"Best val loss: {best_val_loss:.4f}")
        break

if patience_counter < patience:
    print(f"\nCompleted all {max_epochs} epochs without early stopping.")

</details>

---

*End of Notebook 5.*